# 02 | Well-level aggregation, annotation, and control-based normalization

**Purpose.** Aggregate CellProfiler single-cell profiles by plate and well using the median, attach experimental platemap metadata, normalize each plate using its negative-control wells, and select morphological features.

**Confirmed analysis choice:** `mad_robustize` with `Metadata_control_type == 'negcon'` and `mad_robustize_epsilon=0`. The alternative normalization against all samples has been removed.

**Input:** single-cell CSV from Notebook 01 and barcode/platemap metadata. **Output:** well-level feature table with cell counts.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import pycytominer

SCRIPTS_PARENT = Path("../external").resolve()
sys.path.insert(0, str(SCRIPTS_PARENT))
from scripts_notebooks_fossa.profiles import generate_profiles

OUTPUT_DIR = Path("../results").resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Select the input profile and metadata mapping

The upstream Broad Institute helper requests the profile CSV, project identifier, plate-to-platemap barcode table, and metadata root directory. The original interactive selection mechanism is retained here because it defines how these experiment-specific files are located.

In [ ]:
profile, project_name, barcode_df, platemap_path = generate_profiles.profiling_inputs()
single_cells = pd.read_csv(profile)
required = {"Metadata_Plate", "Metadata_Well", "Nuclei_Number_Object_Number"}
missing = required - set(single_cells.columns)
if missing:
    raise KeyError(f"Required columns missing from the single-cell input: {sorted(missing)}")
print(f"Input: {single_cells.shape[0]:,} rows × {single_cells.shape[1]:,} columns")

## 2. Count cells in each well

Count nonmissing `Nuclei_Number_Object_Number` entries for each plate–well combination. In this original workflow, this column is used to count the nucleus-associated rows; its numeric values themselves are **not** treated as cell counts.

In [ ]:
cell_counts = (
    single_cells.groupby(["Metadata_Plate", "Metadata_Well"])["Nuclei_Number_Object_Number"]
    .count().reset_index(name="Metadata_cell_count")
)
print(f"Well counts: {len(cell_counts)}")

## 3. Aggregate single-cell features by well

The median of each morphological feature is computed independently for each plate and well using `pycytominer.aggregate`. This reduces single-cell observations to well-level profiles while preserving plate and well identifiers.

In [ ]:
well_profiles = pycytominer.aggregate(
    single_cells,
    strata=["Metadata_Plate", "Metadata_Well"],
    operation="median",
)
well_profiles = well_profiles.dropna(subset=["Metadata_Plate"])
plate_ids = well_profiles["Metadata_Plate"].unique().tolist()
print(f"Aggregated profiles: {well_profiles.shape}; plates: {plate_ids}")

## 4. Annotate wells using plate-specific platemaps

For each plate, locate the matching `Assay_Plate_Barcode` entry and load its corresponding platemap CSV. Join experimental annotations to the aggregated well profiles through the original `Metadata_well_position` ↔ `Metadata_Well` mapping. A missing or ambiguous barcode now raises an explicit error.

In [ ]:
annotated_plates = []
for plate in plate_ids:
    matches = barcode_df.loc[barcode_df["Assay_Plate_Barcode"] == plate]
    if len(matches) != 1:
        raise ValueError(f"Expected one barcode mapping for {plate}; found {len(matches)}")
    platemap_file = Path(platemap_path) / "platemap" / matches.iloc[0]["Plate_Map_Name"]
    metadata = pd.read_csv(platemap_file)
    plate_profiles = well_profiles.loc[well_profiles["Metadata_Plate"] == plate]
    annotated = pycytominer.annotate(
        profiles=plate_profiles,
        platemap=metadata,
        join_on=["Metadata_well_position", "Metadata_Well"],
    )
    annotated_plates.append(annotated)
    print(f"Annotated {plate}: {annotated.shape}")
annotated_profiles = pd.concat(annotated_plates, ignore_index=True)

## 5. Normalize separately within each plate using negative controls

`pycytominer.normalize` performs MAD-based robust normalization, estimating the location and scale parameters from wells annotated as negative controls (`negcon`). The normalization is fitted separately within each plate, as in the original notebook, and applied to its full set of wells. The epsilon parameter is fixed at zero to reproduce the original call.

In [ ]:
normalized_plates = []
for plate in plate_ids:
    plate_profiles = annotated_profiles.loc[annotated_profiles["Metadata_Plate"] == plate]
    n_controls = (plate_profiles["Metadata_control_type"] == "negcon").sum()
    if n_controls == 0:
        raise ValueError(f"No negative-control wells found for plate {plate}")
    normalized = pycytominer.normalize(
        plate_profiles,
        method="mad_robustize",
        mad_robustize_epsilon=0,
        samples="Metadata_control_type == 'negcon'",
    )
    normalized_plates.append(normalized)
    print(f"Normalized {plate}: {len(plate_profiles)} wells; {n_controls} negative controls")
normalized_profiles = pd.concat(normalized_plates, ignore_index=True)

## 6. Select morphological features

The original four filters are applied in their recorded order: correlation threshold, variance threshold, columns with missing values, and outlier filtering. The original `outlier_cutoff=500` is retained. The number of retained columns is reported for traceability.

In [ ]:
selected_profiles = pycytominer.feature_select(
    normalized_profiles,
    operation=["correlation_threshold", "variance_threshold", "drop_na_columns", "drop_outliers"],
    outlier_cutoff=500,
)
print(f"Columns before selection: {normalized_profiles.shape[1]}")
print(f"Columns after selection: {selected_profiles.shape[1]}")

## 7. Restore cell-count metadata and export

Merge the per-well nucleus-associated object counts back into the selected profiles. Export without the pandas index. This table is used by the downstream LDA and morphological feature analyses.

In [ ]:
well_results = cell_counts.merge(
    selected_profiles,
    on=["Metadata_Plate", "Metadata_Well"],
    how="inner",
    validate="one_to_one",
)
output_file = OUTPUT_DIR / f"{project_name}_normalized_negcon_feature_select.csv"
well_results.to_csv(output_file, index=False)
print(f"Saved {well_results.shape} to {output_file}")